# LiteMedSAM on a Colab GPU — step by step
Run cells from top to bottom. This notebook runs the original authors' inference script on their ultrasound demonstration and supplied bounding box. It does not train a model.

First select **Runtime → Change runtime type → GPU** (T4 if offered), save, and connect. GPU availability depends on your Colab account/session.

Validation status: the same upstream script, checkpoint and demo ran on local CPU. This notebook has not yet been executed in a Colab GPU session. Stop at the first failed cell and inspect the actual error.

## 1. Check that a GPU is available
This must print a GPU name and a successful CUDA tensor operation. Selecting a GPU runtime alone does not move a model to the GPU.

In [ ]:
import sys, subprocess, json, platform
from pathlib import Path
import torch
print("Python:", sys.version)
print("PyTorch:", torch.__version__, "CUDA build:", torch.version.cuda)
assert torch.cuda.is_available(), "No GPU: select Runtime > Change runtime type > GPU, then reconnect."
print("GPU:", torch.cuda.get_device_name(0))
print("CUDA tensor check:", (torch.ones(1, device="cuda") + 1).item())
subprocess.run(["nvidia-smi"], check=True)


**Result/inference:** A successful CUDA check establishes that this session can execute GPU tensor operations. It does not yet establish that LiteMedSAM runs.

## 2. Download your fork and select the LiteMedSAM code
A fork is your GitHub copy. A clone is a local copy inside this Colab runtime. We pin the previously inspected commit so later upstream changes do not silently change the experiment.

In [ ]:
REPO = Path("/content/MedSAM")
COMMIT = "b0fab476e54e631dd412b25e0db9fdf2a2b0f54c"
if not REPO.exists():
    subprocess.run(["git", "clone", "--branch", "LiteMedSAM", "--single-branch",
                    "https://github.com/KrishnachaithanyaThummala/MedSAM.git", str(REPO)], check=True)
assert (REPO/".git").exists(), "Existing directory is not a Git checkout."
subprocess.run(["git", "checkout", "--detach", COMMIT], cwd=REPO, check=True)
actual = subprocess.check_output(["git", "rev-parse", "HEAD"], cwd=REPO, text=True).strip()
assert actual == COMMIT
print("Source commit:", actual)


**Result/inference:** The checkout contains the LiteMedSAM branch code, pinned to a specific source revision. Detached HEAD is intentional for this reproduction notebook.

## 3. Install the inference dependencies
Keep Colab's supplied PyTorch/CUDA pair. Install the packages required by the inference path rather than the entire training environment. These are preparation choices; compatibility is checked by the subsequent imports. If Colab asks you to restart the session after installation, restart and rerun from cell 1.

In [ ]:
subprocess.run([sys.executable, "-m", "pip", "install", "timm==0.9.12", "gdown==5.2.0",
                "opencv-python-headless", "matplotlib", "pandas", "tqdm"], check=True)
# Check the SAME Python environment the inference subprocess will use.
subprocess.run([sys.executable, "-c",
    "import torch, torchvision, timm, cv2, pandas, matplotlib; "
    "from tiny_vit_sam import TinyViT; "
    "from segment_anything.modeling import PromptEncoder, MaskDecoder, TwoWayTransformer; "
    "print('Inference imports OK; CUDA:', torch.cuda.is_available())"], cwd=REPO, check=True)


**Result/inference:** Successful imports establish that the source components and installed libraries can load. Deprecation warnings are different from an exception. Do not continue past an import error.

## 4. Download the official weights and one official demonstration
The checkpoint is learned model weights; it is separate from the source repository. The NPZ file contains an RGB image and supplied box. Links come from the official README/demo folder. Only one demo file is needed.

In [ ]:
import hashlib
import gdown
import numpy as np
CHECKPOINT = REPO/"work_dir/LiteMedSAM/lite_medsam.pth"
INPUT = REPO/"colab_demo_input"
OUTPUT = REPO/"colab_demo_output"
OVERLAY = REPO/"colab_demo_overlay"
INPUT.mkdir(exist_ok=True)
SAMPLE = INPUT/"2DBox_US_demo.npz"
if not CHECKPOINT.exists():
    gdown.download(id="18Zed-TUTsmr2zc5CHUWd5Tu13nb6vq6z", output=str(CHECKPOINT), quiet=False)
expected_hash = "79d8c9dca6db4d69d3f905579e5250af05e859fff9c1f543e89a513c3028ce76"
assert CHECKPOINT.exists(), "Checkpoint download failed; see troubleshooting below."
assert hashlib.sha256(CHECKPOINT.read_bytes()).hexdigest() == expected_hash, "Checkpoint hash differs from locally tested weights."
if not SAMPLE.exists():
    gdown.download(id="1SBsOOXOCavdQbVLb82O0R0vIlS5TqIKg", output=str(SAMPLE), quiet=False)
with np.load(SAMPLE, allow_pickle=False) as sample:
    image, boxes = sample["imgs"], sample["boxes"]
assert image.shape == (420,580,3) and boxes.shape == (1,4)
print("Image:", image.shape, image.dtype, "Box:", boxes.tolist())


**Result/inference:** Expected image shape: 420×580×3. Expected box: [265,78,347,196], in original-image pixels. The checkpoint hash verifies the same file used locally, not model accuracy.

## 5. Run the original inference script on CUDA
`-i` selects the input folder; `-o` selects saved masks; `-lite_medsam_checkpoint_path` selects weights; **`-device cuda:0` moves the upstream model and image tensors to the GPU**. Keep all these arguments explicit.

In [ ]:
command = [sys.executable, "CVPR24_LiteMedSAM_infer.py",
           "-i", str(INPUT), "-o", str(OUTPUT),
           "-lite_medsam_checkpoint_path", str(CHECKPOINT),
           "-device", "cuda:0", "-png_save_dir", str(OVERLAY)]
print("Running:", " ".join(command))
result = subprocess.run(command, cwd=REPO, text=True, stdout=subprocess.PIPE, stderr=subprocess.STDOUT)
print(result.stdout)
OUTPUT.mkdir(exist_ok=True)
(OUTPUT/"inference.log").write_text(result.stdout)
result.check_returncode()
assert (OUTPUT/SAMPLE.name).exists(), "No saved mask was produced."


**Result/inference:** A zero exit code plus a saved prediction establishes the basic forward-pass check. The script's case timing includes more than pure GPU model computation; it is not a controlled GPU benchmark. GPU and CPU masks may differ slightly numerically.

## 6. Inspect and save the results
Show the input, supplied box, binary mask and overlay. Save the mask and provenance; download the archive before ending the runtime.

In [ ]:
import matplotlib.pyplot as plt
from matplotlib.patches import Rectangle
import pandas as pd
with np.load(OUTPUT/SAMPLE.name, allow_pickle=False) as prediction:
    labels = prediction["segs"]
assert labels.shape == image.shape[:2]
assert set(np.unique(labels)).issubset({0,1})
mask = labels > 0
assert np.isfinite(labels).all()
plt.imsave(OUTPUT/"mask.png", mask.astype(np.uint8)*255, cmap="gray", vmin=0, vmax=255)
fig, axes = plt.subplots(1,3,figsize=(15,5))
axes[0].imshow(image); axes[0].set_title("Image + supplied box")
x0,y0,x1,y1 = boxes[0]
axes[0].add_patch(Rectangle((x0,y0),x1-x0,y1-y0,fill=False,edgecolor="cyan",linewidth=2))
axes[1].imshow(mask,cmap="gray"); axes[1].set_title("Predicted mask")
axes[2].imshow(image); axes[2].imshow(np.ma.masked_where(~mask,mask),cmap="autumn",alpha=0.5)
axes[2].set_title("Mask overlay")
for ax in axes: ax.axis("off")
plt.tight_layout(); plt.savefig(OUTPUT/"summary.png",dpi=160); plt.show()
print("Foreground pixels:", int(mask.sum()))
if (OUTPUT/"efficiency.csv").exists(): display(pd.read_csv(OUTPUT/"efficiency.csv"))
report = {"source_commit":actual, "fork":"https://github.com/KrishnachaithanyaThummala/MedSAM",
          "device":"cuda:0", "gpu":torch.cuda.get_device_name(0),
          "python":sys.version, "torch":torch.__version__, "cuda_build":torch.version.cuda,
          "checkpoint_sha256":expected_hash, "box_xyxy":boxes.tolist(),
          "mask_shape":list(mask.shape), "foreground_pixels":int(mask.sum()),
          "command":command, "interpretation":"Software execution check, not clinical accuracy evaluation."}
(OUTPUT/"run.json").write_text(json.dumps(report,indent=2))
(OUTPUT/"requirements-colab.txt").write_text(subprocess.check_output([sys.executable,"-m","pip","freeze"],text=True))


**Result/inference:** The displayed mask and printed foreground count are actual outputs from your session. The local CPU run produced 4,990 foreground pixels; that is a reference observation, not a GPU acceptance threshold or an accuracy score.

In [ ]:
import shutil
from google.colab import files
archive = shutil.make_archive("/content/LiteMedSAM_GPU_results", "zip", root_dir=OUTPUT)
files.download(archive)


**Result/inference:** The ZIP contains the saved NPZ mask, PNG mask, summary plot, logs, timing CSV, environment versions and provenance. Your notebook itself should also be saved in Drive using Colab's File menu.

## What the folders mean
- `tiny_vit_sam.py`: lightweight image encoder.
- `segment_anything/modeling/`: prompt encoder, transformer, mask decoder and shared layers. Keep this package in place; the script imports it.
- `CVPR24_LiteMedSAM_infer.py`: the entry point executed above.
- `work_dir/LiteMedSAM/`: official pretrained checkpoint location.
- `test_demo/`: upstream demo placeholder; this notebook downloads one input into `colab_demo_input/` instead.
- `train_one_gpu.py`, `train_multi_gpus.py`, `distillation/`: learning/fine-tuning; not needed for this pretrained demonstration.
- `pre_*.py`, `npz_to_npy.py`: dataset preparation; not needed for this already prepared demo.
- `evaluation/`: metrics requiring reference annotations; not part of this execution check.
- Docker files: an alternative packaged execution route; not needed here.

## Troubleshooting
- **CUDA unavailable:** select a GPU runtime and reconnect. If Colab cannot allocate one, retry when capacity is available. CPU fallback is deliberately not automatic.
- **Google Drive download quota/permission error:** open the official checkpoint/demo links from the repository README in your browser. Download the checkpoint and `imgs/2DBox_US_demo.npz`, upload them using Colab's Files pane into the paths printed in cell 4, then rerun cell 4. Do not substitute random weights.
- **Import error after package installation:** restart the session and rerun in order. Preserve the exact traceback if it persists; do not replace CUDA/PyTorch versions at random.
- **Missing files after reconnecting:** Colab storage is temporary. Rerun setup/download cells and keep downloaded results on your computer or Drive.

Next study step: open `CVPR24_LiteMedSAM_infer.py` and follow the functions in this order: resize/pad → TinyViT constructor → PromptEncoder constructor → MaskDecoder constructor → `MedSAM_infer_npz_2D` → `medsam_inference`.